# Methods 4 – Word embeddings

## Objective

The first step is to download the pretrained GloVe model. A word vector represents a word using numbers. In this assignment, I will use pretrained GloVe word vectors to explore how different words are related to each other.

In [1]:

from pathlib import Path
from urllib.request import urlretrieve

url = "https://zenodo.org/records/4925376/files/glove.6B.50d.zip?download=1"
zip_path = Path("glove.6B.50d.zip")

if not zip_path.exists():
    urlretrieve(url, zip_path)

print("Downloaded file:", zip_path.resolve())
print("File size (MB):", round(zip_path.stat().st_size / 1_000_000, 1))

Downloaded file: C:\DEV\Metropolia\neuroverkot\neural_networks\assg 4\glove.6B.50d.zip
File size (MB): 58.9


In [2]:

import zipfile

# Extract the GloVe ZIP file
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    print("Files inside ZIP:", zip_ref.namelist())
    zip_ref.extractall()

print("Extraction completed!")


Files inside ZIP: ['glove.6B.50d']
Extraction completed!


## Loading the GloVe embeddings

Here i extracted the Glove embeddings from the zip file. The purpose of this is to access word vectors. 

In [3]:
import numpy as np

word_to_index = {}
vector_dtype = np.dtype("<f4")

with open("glove.6B.50d", "rb") as source:
    word_count, vector_size = map(int, source.readline().split())
    vectors = np.empty((word_count, vector_size), dtype=np.float32)
    vector_bytes = vector_size * vector_dtype.itemsize

    # Each word is followed by a space and a fixed-size binary vector.
    for index in range(word_count):
        word = bytearray()
        while (byte := source.read(1)) != b" ":
            if not byte:
                raise EOFError(f"Missing word at index {index}")
            if byte != b"\n":
                word.extend(byte)

        data = source.read(vector_bytes)
        if len(data) != vector_bytes:
            raise EOFError(f"Incomplete vector at index {index}")
        word_to_index[word.decode("utf-8")] = index
        vectors[index] = np.frombuffer(data, dtype=vector_dtype)

print(f"Loaded {word_count:,} word vectors with {vector_size} dimensions.")

Loaded 400,000 word vectors with 50 dimensions.


As we can see here the model containes 400,000 word vectors, and reach vector has 50 dimentions. Next step is to get the vectors for man, women, and king.

In [5]:
man = vectors[word_to_index["man"]]
woman = vectors[word_to_index["woman"]]
king = vectors[word_to_index["king"]]

print("Man:", man[:5])
print("Woman:", woman[:5])
print("King:", king[:5])

print("Vector length:", len(man))


Man: [-0.094386  0.43007  -0.17224  -0.45529   1.6447  ]
Woman: [-0.18153  0.64827 -0.5821  -0.49451  1.5415 ]
King: [ 0.50451   0.68607  -0.59517  -0.022801  0.60046 ]
Vector length: 50
